In [4]:
# Importing required libraries
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib


In [5]:
netflix_data = pd.read_csv('NetflixDataset.csv', encoding='latin-1')


In [6]:
netflix_data.head()


,Title,Genre,Tags,Languages,Country Availability,Runtime,Director,Writer,Actors,View Rating,...,Awards Nominated For,Boxoffice,Release Date,Netflix Release Date,Production House,Netflix Link,Summary,Series or Movie,IMDb Votes,Image
0,Lets Fight Ghost,"Crime, Drama, Fantasy, Horror, Romance","Comedy Programmes,Romantic TV Comedies,Horror ...","Swedish, Spanish",Thailand,< 30 minutes,Tomas Alfredson,John Ajvide Lindqvist,"Lina Leandersson, Kåre Hedebrant, Per Ragnar, ...",R,...,57.0,"$21,22,065",12-Dec-08,04-03-2021,"Canal+, Sandrew Metronome",https://www.netflix.com/watch/81415947,A med student with a supernatural gift tries t...,Series,205926.0,https://occ-0-4708-64.1.nflxso.net/dnm/api/v6/...
1,HOW TO BUILD A GIRL,Comedy,"Dramas,Comedies,Films Based on Books,British",English,Canada,1-2 hour,Coky Giedroyc,Caitlin Moran,"Cleo, Paddy Considine, Beanie Feldstein, Dónal...",R,...,NaN,"$70,632",08-May-20,04-03-2021,"Film 4, Monumental Pictures, Lionsgate",https://www.netflix.com/watch/81041267,"When nerdy Johanna moves to London, things get...",Movie,2838.0,https://occ-0-1081-999.1.nflxso.net/dnm/api/v6...
2,The Con-Heartist,"Comedy, Romance","Romantic Comedies,Comedies,Romantic Films,Thai...",Thai,Thailand,> 2 hrs,Mez Tharatorn,"Pattaranad Bhiboonsawade, Mez Tharatorn, Thods...","Kathaleeya McIntosh, Nadech Kugimiya, Pimchano...",NaN,...,NaN,NaN,03-Dec-20,03-03-2021,NaN,https://www.netflix.com/watch/81306155,After her ex-boyfriend cons her out of a large...,Movie,131.0,https://occ-0-2188-64.1.nflxso.net/dnm/api/v6/...
3,Gleboka woda,Drama,"TV Dramas,Polish TV Shows,Social Issue TV Dramas",Polish,Poland,< 30 minutes,NaN,NaN,"Katarzyna Maciag, Piotr Nowak, Marcin Dorocins...",NaN,...,4.0,NaN,14-Jun-11,03-03-2021,NaN,https://www.netflix.com/watch/81307527,A group of social welfare workers led by their...,Series,47.0,https://occ-0-2508-2706.1.nflxso.net/dnm/api/v...
4,Only a Mother,Drama,"Social Issue Dramas,Dramas,Movies Based on Boo...",Swedish,"Lithuania,Poland,France,Italy,Spain,Greece,Bel...",1-2 hour,Alf Sjöberg,Ivar Lo-Johansson,"Hugo Björne, Eva Dahlbeck, Ulf Palme, Ragnar F...",NaN,...,1.0,NaN,31-Oct-49,03-03-2021,NaN,https://www.netflix.com/watch/81382068,An unhappily married farm worker struggling to...,Movie,88.0,https://occ-0-2851-41.1.nflxso.net/dnm/api/v6/...


In [8]:
netflix_data.columns


Index(['Title', 'Genre', 'Tags', 'Languages', 'Country Availability',
       'Runtime', 'Director', 'Writer', 'Actors', 'View Rating', 'IMDb Score',
       'Awards Received', 'Awards Nominated For', 'Boxoffice', 'Release Date',
       'Netflix Release Date', 'Production House', 'Netflix Link', 'Summary',
       'Series or Movie', 'IMDb Votes', 'Image'],
      dtype='object')

In [9]:
def combine_features(row):
    return f"{row['Genre']} {row['Tags']} {row['Actors']} {row['Director']} {row['Writer']} {row['Summary']}"


In [10]:
netflix_data['Combined_Features'] = netflix_data.fillna('').apply(combine_features, axis=1)
netflix_data.head(3)


,Title,Genre,Tags,Languages,Country Availability,Runtime,Director,Writer,Actors,View Rating,...,Boxoffice,Release Date,Netflix Release Date,Production House,Netflix Link,Summary,Series or Movie,IMDb Votes,Image,Combined_Features
0,Lets Fight Ghost,"Crime, Drama, Fantasy, Horror, Romance","Comedy Programmes,Romantic TV Comedies,Horror ...","Swedish, Spanish",Thailand,< 30 minutes,Tomas Alfredson,John Ajvide Lindqvist,"Lina Leandersson, Kåre Hedebrant, Per Ragnar, ...",R,...,"$21,22,065",12-Dec-08,04-03-2021,"Canal+, Sandrew Metronome",https://www.netflix.com/watch/81415947,A med student with a supernatural gift tries t...,Series,205926.0,https://occ-0-4708-64.1.nflxso.net/dnm/api/v6/...,"Crime, Drama, Fantasy, Horror, Romance Comedy ..."
1,HOW TO BUILD A GIRL,Comedy,"Dramas,Comedies,Films Based on Books,British",English,Canada,1-2 hour,Coky Giedroyc,Caitlin Moran,"Cleo, Paddy Considine, Beanie Feldstein, Dónal...",R,...,"$70,632",08-May-20,04-03-2021,"Film 4, Monumental Pictures, Lionsgate",https://www.netflix.com/watch/81041267,"When nerdy Johanna moves to London, things get...",Movie,2838.0,https://occ-0-1081-999.1.nflxso.net/dnm/api/v6...,"Comedy Dramas,Comedies,Films Based on Books,Br..."
2,The Con-Heartist,"Comedy, Romance","Romantic Comedies,Comedies,Romantic Films,Thai...",Thai,Thailand,> 2 hrs,Mez Tharatorn,"Pattaranad Bhiboonsawade, Mez Tharatorn, Thods...","Kathaleeya McIntosh, Nadech Kugimiya, Pimchano...",NaN,...,NaN,03-Dec-20,03-03-2021,NaN,https://www.netflix.com/watch/81306155,After her ex-boyfriend cons her out of a large...,Movie,131.0,https://occ-0-2188-64.1.nflxso.net/dnm/api/v6/...,"Comedy, Romance Romantic Comedies,Comedies,Rom..."


In [11]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(stop_words='english')
count_matrix = cv.fit_transform(netflix_data['Combined_Features'])

print("✅ Count matrix shape:", count_matrix.shape)


✅ Count matrix shape: (9403, 43713)


In [12]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(count_matrix)
print("✅ Cosine similarity matrix shape:", cosine_sim.shape)


✅ Cosine similarity matrix shape: (9403, 9403)


In [13]:
netflix_data = netflix_data.reset_index()

def recommend_movie(title, cosine_sim=cosine_sim):
    title = title.strip().title()
    if title not in netflix_data['Title'].values:
        return ["Sorry, this title is not found in the dataset."]
    
    idx = netflix_data[netflix_data['Title'] == title].index[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    top_movies = [netflix_data.iloc[i[0]]['Title'] for i in sim_scores[1:6]]
    return top_movies


In [14]:
recommend_movie("Inception")


['Lucy',
 'Blade Runner 2049',
 'Minority Report',
 'The Dark Knight Rises',
 'RoboCop']

In [15]:
import joblib

joblib.dump(netflix_data, 'netflix_data.pkl')
joblib.dump(cosine_sim, 'cosine_sim.pkl')

print("✅ Model and data saved successfully!")


✅ Model and data saved successfully!


In [16]:
recommend_movie("The Dark Knight Rises")

['Batman Begins', 'Inception', 'Batman Returns', 'Man of Steel', 'RoboCop']